In [0]:
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

pit_stops_table = f"{catalog}.f1_silver.pit_stops"

pit_stops_df = spark.table(pit_stops_table)

pit_performance_df = (
    pit_stops_df
    .groupBy(
        "race_id",
        "driver_id"
    )
    .agg(
        F.count("*").alias("pit_stop_count"),
        F.avg("duration_ms").alias("average_stop_duration_ms"),
        F.min("duration_ms").alias("fastest_stop_duration_ms"),
        F.max("duration_ms").alias("slowest_stop_duration_ms"),
        F.min("pit_lap").alias("first_pit_lap"),
        F.max("pit_lap").alias("last_pit_lap")
    )
)

display(pit_performance_df)

In [0]:
pit_gold_table = f"{catalog}.f1_gold.pit_stop_performance"

(
    pit_performance_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(pit_gold_table)
)

print(f"Saved: {pit_gold_table}")